# 05A — Feature Extraction (Sliding Window)

**Stage 5A.** Extract physiological features from windowed WESAD signals using **sliding windows**.

Produces the **same wide-format feature set** as `05B_feature_extraction_forked.ipynb` (one row per window, all signals combined), but uses overlapping sliding windows instead of non-overlapping ones.

### Features extracted per window

- **Window statistics** (mean, std, min, max) for every signal channel:
  - Wrist: EDA, EDA_SCR, EDA_SCL, BVP, TEMP, ACC_x/y/z, net_acc
  - Chest: ECG, EDA_C, EDA_SCR_C, EDA_SCL_C, ACC_x_C/y_C/z_C, net_acc_C, EMG_C, Resp_C, Temp_C
- **ECG**: BPM, IBI, SDNN, SDSD, RMSSD, PNN20, PNN50 (via `heartpy`).
- **ACC**: Net acceleration, absolute integrals, peak frequencies (wrist & chest).
- **EDA**: Slope, dynamic range (wrist & chest).
- **EMG**: Dynamic range, absolute integral.
- **Resp**: Respiration rate, inhalation/exhalation durations, I/E ratio (via `biosppy`).
- **TEMP**: Dynamic range, slope (wrist & chest).
- **BVP**: Peak frequency.

Input: `data/preprocessed/WESAD/S{id}_clean.pkl`  
Output: `Data/features_c/S{id}_features.csv`

In [9]:
import sys, gc, time
from pathlib import Path
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Scripts' / 'Preprocess').exists()) / 'Scripts' / 'Preprocess'
sys.path.insert(0, str(HERE))

import numpy as np
import pandas as pd
import wesad_preprocess as wp

# Window parameters (in seconds)
WINDOW_SEC = 60.0   # same for all signals (matches 05B)
SHIFT_SEC = 20.0

SUBJECTS = wp.SUBJECT_IDS

## 1. Windowing & Feature Extraction Functions

In [10]:
import matplotlib
matplotlib.use('Agg')
import scipy.signal as scisig
import scipy.stats
import heartpy as hp
import warnings
warnings.simplefilter("ignore")

# Stub out biosppy's interactive plotting (requires tkinter, which is not installed)
import sys
from unittest.mock import MagicMock
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.ecg'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

def extract_sliding_windows(arr, label_arr, fs, window_sec, shift_sec):
    arr = np.asarray(arr)
    label_arr = np.asarray(label_arr)
    win_len = int(round(window_sec * fs))
    shift_len = int(round(shift_sec * fs))
    if len(arr) < win_len:
        return np.array([]), np.array([])
    n_windows = (len(arr) - win_len) // shift_len + 1
    shape = (n_windows, win_len)
    strides = (arr.strides[0] * shift_len, arr.strides[0])
    if arr.ndim == 1:
        windows = np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)
    else:
        n_channels = arr.shape[1]
        shape_multi = (n_windows, win_len, n_channels)
        strides_multi = (arr.strides[0] * shift_len, arr.strides[0], arr.strides[1])
        windows = np.lib.stride_tricks.as_strided(arr, shape=shape_multi, strides=strides_multi)
    label_shape = (n_windows, win_len)
    label_strides = (label_arr.strides[0] * shift_len, label_arr.strides[0])
    win_labels = np.lib.stride_tricks.as_strided(label_arr, shape=label_shape, strides=label_strides)
    labels = (win_labels.mean(axis=1) >= 0.5).astype(int)
    return windows, labels

def get_window_stats(data, label=-1):
    return {'mean': np.mean(data), 'std': np.std(data), 'min': np.amin(data), 'max': np.amax(data), 'label': label}

def get_net_accel(data, part):
    if part == 'wrist':
        return np.sqrt(data['ACC_x']**2 + data['ACC_y']**2 + data['ACC_z']**2)
    else:
        return np.sqrt(data['ACC_x_C']**2 + data['ACC_y_C']**2 + data['ACC_z_C']**2)

def get_slope(series):
    return scipy.stats.linregress(np.arange(len(series)), series)[0]

def get_absolute_integral(x):
    return np.sum(np.abs(x))

def get_dynamic_range(x):
    return np.max(x) / np.min(x)

def get_peak_freq(x):
    f, Pxx = scisig.periodogram(x, fs=8)
    psd_dict = {amp: freq for amp, freq in zip(Pxx, f)}
    return psd_dict[max(psd_dict.keys())]

def get_window_stats_ecg(data, label=-1):
    wd, m = hp.process(data['ECG'].dropna().reset_index(drop=True), 700)
    return {'bpm': m['bpm'], 'ibi': m['ibi'], 'sdnn': m['sdnn'], 'sdsd': m['sdsd'],
            'rmssd': m['rmssd'], 'pnn20': m['pnn20'], 'pnn50': m['pnn50']}

def get_resp_features(resp_data):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resp_rate, filtered, zeros, resp_rate_ts, resp_rate = biosppy.signals.resp.resp(resp_data, sampling_rate=700, show=False)
        extremas, values = biosppy.signals.tools.find_extrema(signal=filtered, mode='both')
    inhal_durations = []
    exhal_durations = []
    last_index = 0
    for i in range(len(extremas)):
        if values[i] * values[last_index] < 0:
            if values[last_index] < 0:
                inhal_durations.append((extremas[i] - extremas[last_index]) / 700)
            else:
                exhal_durations.append((extremas[i] - extremas[last_index]) / 700)
            last_index = i
    inhal_mean = np.mean(inhal_durations) if len(inhal_durations) > 0 else 0
    inhal_std = np.std(inhal_durations) if len(inhal_durations) > 0 else 0
    exhal_mean = np.mean(exhal_durations) if len(exhal_durations) > 0 else 0
    exhal_std = np.std(exhal_durations) if len(exhal_durations) > 0 else 0
    sum_exhal = np.sum(exhal_durations)
    ie_ratio = np.sum(inhal_durations) / sum_exhal if sum_exhal > 0 else 0
    return np.mean(resp_rate), inhal_mean, inhal_std, exhal_mean, exhal_std, ie_ratio

## 2. Feature Extraction per Window

In [11]:
def extract_features_for_subject(clean_data, window_sec=WINDOW_SEC, shift_sec=SHIFT_SEC):
    """
    Extract wide-format features for all sliding windows of a subject.
    Produces the exact same columns, names, and order as 05B (fork-compatible)
    but using overlapping sliding windows.
    """
    SIGNAL_MAP = {
        ('wrist', 'EDA'):     'EDA',
        ('wrist', 'EDA_SCR'): 'EDA_SCR',
        ('wrist', 'EDA_SCL'): 'EDA_SCL',
        ('wrist', 'BVP'):     'BVP',
        ('wrist', 'TEMP'):    'TEMP',
        ('chest', 'ECG'):     'ECG',
        ('chest', 'EDA'):     'EDA_C',
        ('chest', 'EDA_SCR'): 'EDA_SCR_C',
        ('chest', 'EDA_SCL'): 'EDA_SCL_C',
        ('chest', 'EMG'):     'EMG_C',
        ('chest', 'Resp'):    'Resp_C',
        ('chest', 'Temp'):    'Temp_C',
    }

    signal_windows = {}
    label_windows = None
    n_windows = None

    for loc in ('chest', 'wrist'):
        for name, arr in clean_data['signal'][loc].items():
            fs = clean_data['fs'][loc][name]
            lab = clean_data['label'][loc][name]

            wins, win_labs = extract_sliding_windows(arr, lab, fs, window_sec, shift_sec)
            if len(wins) == 0:
                continue

            if n_windows is None:
                n_windows = len(wins)
            else:
                n_windows = min(n_windows, len(wins))

            if label_windows is None:
                label_windows = win_labs

            if name == 'ACC':
                if loc == 'wrist':
                    signal_windows['ACC_x'] = wins[:, :, 0]
                    signal_windows['ACC_y'] = wins[:, :, 1]
                    signal_windows['ACC_z'] = wins[:, :, 2]
                else:
                    signal_windows['ACC_x_C'] = wins[:, :, 0]
                    signal_windows['ACC_y_C'] = wins[:, :, 1]
                    signal_windows['ACC_z_C'] = wins[:, :, 2]
            else:
                col_name = SIGNAL_MAP.get((loc, name))
                if col_name is not None:
                    signal_windows[col_name] = wins

    if n_windows is None or n_windows == 0:
        return pd.DataFrame()

    for key in signal_windows:
        signal_windows[key] = signal_windows[key][:n_windows]
    label_windows = label_windows[:n_windows]

    if 'ACC_x' in signal_windows:
        signal_windows['net_acc'] = np.sqrt(
            signal_windows['ACC_x']**2 + signal_windows['ACC_y']**2 + signal_windows['ACC_z']**2
        )
    if 'ACC_x_C' in signal_windows:
        signal_windows['net_acc_C'] = np.sqrt(
            signal_windows['ACC_x_C']**2 + signal_windows['ACC_y_C']**2 + signal_windows['ACC_z_C']**2
        )

    stat_cols = [
        'net_acc_C', 'net_acc', 'EDA', 'EDA_SCR', 'EDA_SCL', 'BVP', 'TEMP',
        'ACC_x', 'ACC_y', 'ACC_z', 'label', 'ECG', 'EDA_C', 'EDA_SCR_C',
        'EDA_SCL_C', 'ACC_x_C', 'ACC_y_C', 'ACC_z_C', 'EMG_C', 'Resp_C', 'Temp_C'
    ]

    samples = []
    for i in range(n_windows):
        lab = int(label_windows[i])
        
        window_dict = {}
        for col in stat_cols:
            if col == 'label':
                window_dict['label'] = np.full(signal_windows['EDA'].shape[1], lab)
            elif col in signal_windows:
                window_dict[col] = signal_windows[col][i]
        
        # Enforce consistent lengths in window_dict before creating DataFrame
        min_length = min(len(series) for series in window_dict.values())
        for key in window_dict:
            window_dict[key] = window_dict[key][:min_length]  # Truncate to min length

        print("Lengths of each series in window_dict before DataFrame creation:")
        for key, value in window_dict.items():
            print(f"{key}: {len(value)}")
        window = pd.DataFrame(window_dict)

        stats = get_window_stats(window, label=lab)
        ecg_stats = get_window_stats_ecg(window, label=lab)
        row = {}
        for feature_name, values in stats.items():
            if feature_name == 'label':
                continue
            if isinstance(values, pd.Series):
                row.update({f'{column}_{feature_name}': value for column, value in values.items()})
        row['label'] = lab

        row['BVP_peak_freq'] = get_peak_freq(window['BVP'].dropna()) if 'BVP' in window else None
        row['net_acc_abs_integral'] = get_absolute_integral(window['net_acc'].dropna()) if 'net_acc' in window else None
        row['ACC_x_abs_integral'] = get_absolute_integral(window['ACC_x'].dropna()) if 'ACC_x' in window else None
        row['ACC_y_abs_integral'] = get_absolute_integral(window['ACC_y'].dropna()) if 'ACC_y' in window else None
        row['ACC_z_abs_integral'] = get_absolute_integral(window['ACC_z'].dropna()) if 'ACC_z' in window else None
        row['net_acc_C_abs_integral'] = get_absolute_integral(window['net_acc_C'].dropna()) if 'net_acc_C' in window else None
        row['ACC_x_C_abs_integral'] = get_absolute_integral(window['ACC_x_C'].dropna()) if 'ACC_x_C' in window else None
        row['ACC_y_C_abs_integral'] = get_absolute_integral(window['ACC_y_C'].dropna()) if 'ACC_y_C' in window else None
        row['ACC_z_C_abs_integral'] = get_absolute_integral(window['ACC_z_C'].dropna()) if 'ACC_z_C' in window else None
        row['ACC_x_peak_freq'] = get_peak_freq(window['ACC_x'].dropna()) if 'ACC_x' in window else None
        row['ACC_y_peak_freq'] = get_peak_freq(window['ACC_y'].dropna()) if 'ACC_y' in window else None
        row['ACC_z_peak_freq'] = get_peak_freq(window['ACC_z'].dropna()) if 'ACC_z' in window else None
        row['ACC_x_C_peak_freq'] = None if ('ACC_x_C' not in window or window['ACC_x_C'].dropna().empty) else get_peak_freq(window['ACC_x_C'].dropna())
        row['ACC_y_C_peak_freq'] = None if ('ACC_y_C' not in window or window['ACC_y_C'].dropna().empty) else get_peak_freq(window['ACC_y_C'].dropna())
        row['ACC_z_C_peak_freq'] = None if ('ACC_z_C' not in window or window['ACC_z_C'].dropna().empty) else get_peak_freq(window['ACC_z_C'].dropna())
        
        row.update({f'ECG_{key}': value for key, value in ecg_stats.items()})
        
        row['EDA_slope'] = get_slope(window['EDA'].dropna()) if 'EDA' in window else None
        row['EDA_C_slope'] = None if ('ACC_z_C' not in window or window['ACC_z_C'].dropna().empty or 'EDA_C' not in window) else get_slope(window['EDA_C'].dropna())
        row['EDA_drange'] = get_dynamic_range(window['EDA'].dropna()) if 'EDA' in window else None
        row['EDA_C_drange'] = None if ('EDA_C' not in window or window['EDA_C'].dropna().empty) else get_dynamic_range(window['EDA_C'].dropna())
        row['EMG_drange'] = get_dynamic_range(window['EMG_C'].dropna()) if 'EMG_C' in window else None
        row['EMG_abs_integral'] = get_absolute_integral(window['EMG_C'].dropna()) if 'EMG_C' in window else None
        
        if 'Resp_C' in window and not window['Resp_C'].dropna().empty:
            try:
                resp_features = get_resp_features(window['Resp_C'].dropna())
                row.update(dict(zip([
                    'Resp_C_rate', 'Resp_C_Inhal_mean', 'Resp_C_Inhal_std',
                    'Resp_C_Exhal_mean', 'Resp_C_Exhal_std', 'Resp_C_I/E',
                ], resp_features)))
            except Exception:
                pass

        row['TEMP_drange'] = get_dynamic_range(window['TEMP'].dropna()) if 'TEMP' in window else None
        row['TEMP_C_drange'] = None if ('Temp_C' not in window or window['Temp_C'].dropna().empty) else get_dynamic_range(window['Temp_C'].dropna())
        row['TEMP_slope'] = get_slope(window['TEMP'].dropna()) if 'TEMP' in window else None
        row['TEMP_C_slope'] = None if ('Temp_C' not in window or window['Temp_C'].dropna().empty) else get_slope(window['Temp_C'].dropna())
        
        samples.append(row)

    return pd.DataFrame(samples)

## 3. Single-Subject Walkthrough

In [12]:
clean = wp.load_clean_subject(SUBJECTS[0])
features_df = extract_features_for_subject(clean)
print(f"Extracted {len(features_df)} windows x {len(features_df.columns)} features for subject {clean['subject']}")
print(f"Label distribution:\n{features_df['label'].value_counts().to_dict()}")
features_df.head()

Lengths of each series in window_dict before DataFrame creation:
net_acc_C: 42000
net_acc: 1920
EDA: 240
EDA_SCR: 240
EDA_SCL: 240
BVP: 3840
TEMP: 240
ACC_x: 1920
ACC_y: 1920
ACC_z: 1920
label: 240
ECG: 42000
EDA_C: 42000
EDA_SCR_C: 42000
EDA_SCL_C: 42000
ACC_x_C: 42000
ACC_y_C: 42000
ACC_z_C: 42000
EMG_C: 42000
Resp_C: 42000
Temp_C: 42000


ValueError: All arrays must be of the same length

## 4. Process & Save All Subjects

In [ ]:
from wakepy import keep

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
FEATURE_OUTPUT_DIR = REPO_ROOT / 'Data' / 'features_c'
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SKIP_EXTRACTED_SUBJECT = False   # True: skip subjects whose S{id}_features.csv already exists

summary = []
with keep.running():
    for sid in SUBJECTS:
        out_path = FEATURE_OUTPUT_DIR / f"S{sid}_features.csv"

        # Skip already-extracted subjects (e.g. resume after an interrupted run)
        if SKIP_EXTRACTED_SUBJECT and out_path.exists():
            print(f"S{sid}: already extracted -> skip")
            continue

        t0 = time.time()
        clean = wp.load_clean_subject(sid)
        features_df = extract_features_for_subject(clean)

        features_df.to_csv(out_path, index=False)

        summary.append({
            'subject': sid,
            'n_rows': len(features_df),
            'n_features': len(features_df.columns),
            'saved_to': out_path.name,
            'seconds': round(time.time() - t0, 2)
        })
        print(summary[-1])
        del clean, features_df; gc.collect()

pd.DataFrame(summary)

{'subject': 2, 'n_rows': 104, 'n_features': 119, 'saved_to': 'S2_features.csv', 'seconds': 14.48}
{'subject': 3, 'n_rows': 105, 'n_features': 119, 'saved_to': 'S3_features.csv', 'seconds': 19.65}
{'subject': 4, 'n_rows': 106, 'n_features': 119, 'saved_to': 'S4_features.csv', 'seconds': 12.61}
{'subject': 5, 'n_rows': 108, 'n_features': 119, 'saved_to': 'S5_features.csv', 'seconds': 15.9}
{'subject': 6, 'n_rows': 108, 'n_features': 119, 'saved_to': 'S6_features.csv', 'seconds': 19.18}
{'subject': 7, 'n_rows': 107, 'n_features': 119, 'saved_to': 'S7_features.csv', 'seconds': 12.76}
{'subject': 8, 'n_rows': 108, 'n_features': 119, 'saved_to': 'S8_features.csv', 'seconds': 13.56}
{'subject': 9, 'n_rows': 107, 'n_features': 119, 'saved_to': 'S9_features.csv', 'seconds': 15.52}
{'subject': 10, 'n_rows': 111, 'n_features': 119, 'saved_to': 'S10_features.csv', 'seconds': 15.95}
{'subject': 11, 'n_rows': 109, 'n_features': 119, 'saved_to': 'S11_features.csv', 'seconds': 17.54}
{'subject': 13, '

,subject,n_rows,n_features,saved_to,seconds
0,2,104,119,S2_features.csv,14.48
1,3,105,119,S3_features.csv,19.65
2,4,106,119,S4_features.csv,12.61
3,5,108,119,S5_features.csv,15.90
4,6,108,119,S6_features.csv,19.18
5,7,107,119,S7_features.csv,12.76
6,8,108,119,S8_features.csv,13.56
7,9,107,119,S9_features.csv,15.52
8,10,111,119,S10_features.csv,15.95
9,11,109,119,S11_features.csv,17.54


## 5. Inspect Extracted Features

In [ ]:
# Load one extracted feature file to inspect
sample_file = next(FEATURE_OUTPUT_DIR.glob("S*_features.csv"), None)
if sample_file is None:
    raise FileNotFoundError("No extracted features found in FEATURE_OUTPUT_DIR. Run Section 5 first.")

features_df = pd.read_csv(sample_file)
print(f"Inspecting: {sample_file.name} ({len(features_df)} rows, {len(features_df.columns)} columns)")
print(f"Feature distribution by class labels:\n{features_df['label'].value_counts()}")

# Show column names/features
print(f"Features extracted: {', '.join(features_df.columns)}")
features_df.head()

Inspecting: S10_features.csv (111 rows, 119 columns)
Feature distribution by class labels:
0    77
1    34
Name: label, dtype: int64
Features extracted: ACC_x_C_mean, ACC_x_C_std, ACC_x_C_min, ACC_x_C_max, ACC_y_C_mean, ACC_y_C_std, ACC_y_C_min, ACC_y_C_max, ACC_z_C_mean, ACC_z_C_std, ACC_z_C_min, ACC_z_C_max, ECG_mean, ECG_std, ECG_min, ECG_max, EMG_C_mean, EMG_C_std, EMG_C_min, EMG_C_max, EDA_C_mean, EDA_C_std, EDA_C_min, EDA_C_max, EDA_SCR_C_mean, EDA_SCR_C_std, EDA_SCR_C_min, EDA_SCR_C_max, EDA_SCL_C_mean, EDA_SCL_C_std, EDA_SCL_C_min, EDA_SCL_C_max, Temp_C_mean, Temp_C_std, Temp_C_min, Temp_C_max, Resp_C_mean, Resp_C_std, Resp_C_min, Resp_C_max, ACC_x_mean, ACC_x_std, ACC_x_min, ACC_x_max, ACC_y_mean, ACC_y_std, ACC_y_min, ACC_y_max, ACC_z_mean, ACC_z_std, ACC_z_min, ACC_z_max, BVP_mean, BVP_std, BVP_min, BVP_max, EDA_mean, EDA_std, EDA_min, EDA_max, EDA_SCR_mean, EDA_SCR_std, EDA_SCR_min, EDA_SCR_max, EDA_SCL_mean, EDA_SCL_std, EDA_SCL_min, EDA_SCL_max, TEMP_mean, TEMP_std, TEMP_

,ACC_x_C_mean,ACC_x_C_std,ACC_x_C_min,ACC_x_C_max,ACC_y_C_mean,ACC_y_C_std,ACC_y_C_min,ACC_y_C_max,ACC_z_C_mean,ACC_z_C_std,...,Resp_C_Inhal_mean,Resp_C_Inhal_std,Resp_C_Exhal_mean,Resp_C_Exhal_std,Resp_C_I/E,TEMP_drange,TEMP_slope,TEMP_C_drange,TEMP_C_slope,label
0,0.000608,0.000007,0.000545,0.000700,0.000764,0.000017,0.000619,0.000871,0.000778,0.000033,...,1.947143,0.310904,1.896476,0.230302,1.026716,1.003310,0.000288,1.008774,0.000003,0
1,0.000602,0.000010,0.000545,0.000700,0.000748,0.000026,0.000619,0.000871,0.000732,0.000067,...,2.012571,0.530272,1.847238,0.207785,1.089503,1.001502,0.000059,1.009849,0.000003,0
2,0.000597,0.000010,0.000545,0.000700,0.000734,0.000024,0.000619,0.000871,0.000690,0.000063,...,1.997905,0.544213,1.822952,0.232349,1.095972,1.000901,0.000040,1.010340,0.000004,0
3,0.000593,0.000004,0.000579,0.000605,0.000721,0.000004,0.000703,0.000756,0.000655,0.000013,...,1.895804,0.558110,1.738214,0.226925,1.090662,1.000901,-0.000005,1.010335,0.000004,0
4,0.000593,0.000003,0.000580,0.000605,0.000722,0.000005,0.000702,0.000756,0.000657,0.000011,...,1.682857,0.109467,1.672222,0.133400,0.950451,1.001502,-0.000102,1.009063,0.000003,0
